# Feature Detection 

In this notebook a trained fruit classifier is used to perform the Feature detection. The ML model used to make this classifier is MobileNets. In brief, the image is predicted, and the bounding boxes are drawn with out annotations without annotation. 

### All the essential and required libraries are imported

In [202]:
# Import necessary libraries
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from sklearn.preprocessing import LabelBinarizer
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
import cv2
from PIL import Image, ImageEnhance



### Loading and preprocessing the Dataset

Initially the modal is loaded to pridict test images.And the Linear_Binarizer is given with list of classes to which the pridicted images are to be assigned.

In [ ]:
# Load the trained model
model_path = 'D:/Feature Detection/feature_detection/fruit_classifier.h5'  # Replace with the actual path to your saved model
model = load_model(model_path)

classes = ["apple fruit", "banana fruit", "cherry fruit", "chickoo fruit", "grapes fruit", "kiwi fruit", "mango fruit", "orange fruit", "strawberry fruit"]

lb = LabelBinarizer()
lb.fit(classes)

All the images are in different dimentions, so the images are bing preprocessed befor loading into the model.

The images are resised to 225x225 dimensions. This dimension is an arbitrary number, and other reason to choose number is that its a small value and easy to process image. The preprocessed image displayed below a view on preprocessing an image.

In [ ]:
# Load and preprocess an image
image_path = 'D:/Feature Detection/test folder/test_img (4).jpg'  # Replace with the actual path to your image
actual_image=load_img(image_path, target_size=(224, 224))
image = actual_image
image = img_to_array(image)
image = preprocess_input(image)
image = np.expand_dims(image, axis=0)
img = cv2.imread(image_path)
img = cv2.resize(img, (224, 224))
plt.matshow(actual_image)
plt.title("Test Image")
plt.show()


Now the loaded is going to be pridicted in the folloing code cell. This pridicted class will be displayed, in the final images.

In [ ]:
# Get the features from the dense layer
#preds = feature_model.predict(image)
pred_label = lb.classes_[np.argmax(model.predict(image))]

Here comes the intereting part, a function is defined below, which gives a visual representation of what the eyes of the Neural Network see.

This function, 'make_gradcam_heatmap', is designed to create a Grad-CAM (Gradient-weighted Class Activation Mapping) heatmap. Grad-CAM is a popular technique used to visualize which regions of an input image are most important for the predictions made by a convolutional neural network (CNN). In this function, it processes the gradients of a specific layer in the model to highlight the areas of the image that contributed most to the prediction. 

In [207]:

# Function to create Grad-CAM heatmap

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.models.Model(
        [model.inputs], 
        [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        prediction_output = preds[1] if len(preds) > 1 else preds[0]

        if pred_index is None:
            pred_index = tf.argmax(prediction_output[0])
            pred_index = pred_index.numpy()
            if isinstance(pred_index, np.ndarray) and pred_index.size == 1:
                pred_index = int(pred_index)

        class_channel = prediction_output[:, pred_index]

    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    heatmap = np.uint8(255 * heatmap)
    
    return heatmap



To view all the layers of the neural network, this command is used.

In [208]:
#model.summary()

In [ ]:
# Generate the heatmap
last_conv_layer_name = "block_13_expand_BN"  # Replace with the name of your last convolutional layer
heatmap = make_gradcam_heatmap(image, model, last_conv_layer_name)

# Display the heatmap
plt.matshow(heatmap, cmap='gray')
plt.show()

The heatmap resolution is 7x7 which is very low for detecting the edges. In order to detect the edges properly, the resolution of the heatmap image is increases to 224x224 (because the fed to the model are resied to 224x224 during preprocessing, and increasing resolution to this before mentiioned values will enable superimposition)

In [ ]:
# Convert heatmap to PIL Image
heatmap_img = Image.fromarray(heatmap)

resized_heatmap_img = heatmap_img.resize((224, 224), Image.NEAREST)

# Convert the resized image back to a NumPy array
resized_heatmap_array = np.array(resized_heatmap_img)

# Print the shape to confirm it's 224x224
print("Shape of the resized heatmap array:", resized_heatmap_array.shape)

# Display the resized heatmap image
plt.figure(figsize=(5, 5))
plt.title("Resized Enhanced Heatmap (224x224)")
plt.imshow(resized_heatmap_img, cmap='gray')
plt.show()


Locating the brightest spots on the feature map

In [ ]:
max_whites = np.max(resized_heatmap_array)
indices = np.where(resized_heatmap_array == max_whites)

print("Img to array: ", resized_heatmap_array.shape)
print("Maximum value: ", max_whites)
print("Indices: ", indices)

print("Rows of Kernel location: ", indices[0].shape)
print("Columns of Kernel location: ", indices[1].shape)
coordinates = list(zip(indices[1], indices[0]))



Taking all the coordinates of the brightest pixel locations and extracting the kernel from the image

In [ ]:
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
#grayImage_array = np.array(grayImage)

kernel_width = np.max(indices[0]) - np.min(indices[0])
kernel_height = np.max(indices[1]) - np.min(indices[1])

x, y, width, height = np.min(indices[1]), np.min(indices[0]), kernel_width, kernel_height

kernel = img[y:y+height, x:x+width]

plt.matshow(kernel)
plt.title("Kernel")
plt.show()


Template matching ios performed here

In [ ]:
img_test = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
kernel_test = cv2.cvtColor(kernel, cv2.COLOR_BGR2RGB)
#kernel_test = cv2.cvtColor(resized_image, cv2.COLOR_BGR2RGB)

# Perform template matching
result = cv2.matchTemplate(img_test, kernel_test, cv2.TM_SQDIFF)

# Find the location of the best match
min_val, max_val, min_loc, max_loc = cv2.minMaxLoc(result)

threshold = min_val + (max_val - min_val) * 0.091  # Use 20% above the minimum as a threshold
# Find locations where the result is less than or equal to the threshold
locations = np.where(result <= threshold)

# Draw rectangles on matches in the test image
for pt in zip(*locations[::-1]):  # Swap columns and rows
    cv2.rectangle(img_test, pt, (pt[0] + kernel.shape[1], pt[1] + kernel.shape[0]), (0, 255, 0), 2)

# Display the result
plt.imshow(cv2.cvtColor(img_test, cv2.COLOR_BGR2RGB))
plt.title("Detected Features in Test Image")
plt.show()


Drawing the Bounding Boxes

In [ ]:
detection = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

x_min, y_min = float('inf'), float('inf')
x_max, y_max = 0, 0

for pt in zip(*locations[::-1]):  # Swap columns and rows
    x, y = pt
    x_min = min(x_min, x)
    y_min = min(y_min, y)
    x_max = max(x_max, x + kernel.shape[1])
    y_max = max(y_max, y + kernel.shape[0])

# Draw a single rectangle that encloses all matching regions
cv2.rectangle(detection, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)

# Display the result with a single enclosing rectangle
plt.imshow(cv2.cvtColor(detection, cv2.COLOR_BGR2RGB))
plt.title("Detected Features in Test Image (Single Bounding Rectangle)")
plt.axis('off')
plt.show()

In [215]:
# need to use k-means